# Tutorial 3: Protein Embeddings

Protein-language-model embeddings are generated with `BioEmbedder.embed(...)`.
The same call handles gene symbols, UniProt-like protein identifiers, raw
amino-acid sequences, canonical proteins, and isoform-level outputs.


In [ ]:
from embpy import BioEmbedder

RUN_EMBEDDING = False  # Set True when you are ready to run model inference.
embedder = BioEmbedder(device="auto", organism="human")
print(f"device={embedder.device}")


## 1. Gene symbols through ESM-2


In [ ]:
genes = ["TP53", "BRCA1", "EGFR", "KRAS", "MYC"]

if RUN_EMBEDDING:
    esm_adata = embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="esm2_650M",
        output="anndata",
        key="X_esm2_650M",
        pooling_strategy="mean",
    )
    print(esm_adata.varm["X_esm2_650M"].shape)
    print(esm_adata.uns["embeddings"]["X_esm2_650M"]["provenance"])


## 2. Raw amino-acid sequences


In [ ]:
p53_fragment = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQK"
)

if RUN_EMBEDDING:
    seq_adata = embedder.embed(
        [p53_fragment],
        entity_type="sequence",
        model="esm2_650M",
        output="anndata",
        key="X_esm2_sequence",
        pooling_strategy="mean",
    )
    print(seq_adata.obsm["X_esm2_sequence"].shape)


## 3. Canonical protein and isoform payloads


In [ ]:
if RUN_EMBEDDING:
    canonical = embedder.embed(
        ["TP53"],
        entity_type="protein",
        id_type="symbol",
        model="esm2_650M",
        output="payload",
        key="X_tp53_canonical_esm2",
        isoform="canonical",
    )
    isoforms = embedder.embed(
        ["TP53"],
        entity_type="protein",
        id_type="symbol",
        model="esm2_650M",
        output="payload",
        key="X_tp53_isoforms_esm2",
        isoform="all",
    )
    print(canonical["entity_type"], canonical["n_entities"])
    print(isoforms["entity_type"], isoforms["n_entities"])


## 4. Compare protein model families with one API


In [ ]:
if RUN_EMBEDDING:
    protein_models = ["esm2_150M", "esm2_650M", "prot_t5_xl_half"]
    comparison = embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model=protein_models,
        output="anndata",
        harmonize_dim=128,
    )
    print(list(comparison.varm.keys()))


## 5. Export a reusable protein table


In [ ]:
if RUN_EMBEDDING:
    embedder.embed(
        genes,
        entity_type="gene",
        id_type="symbol",
        model="esm2_650M",
        output="table",
        path="protein_embeddings_esm2_650M.csv",
        fmt="csv",
    )
